## Section 1: The Dataset
For this project, I chose a dataset on student depression that I found on Kaggle. The dataset includes responses from students about factors of their lives that might contribute to depression such as academic pressure and lifestyle habits. I chose this dataset because mental health, especially among students, is a hot topic of discussion right now and something I care deeply about. I discovered this dataset in Kaggle by searching for different datasets on students and finding one with solid credibility and reviews (key note: 0/1 corresponds to yes/no in Depression column). Using Data Science I want to explore (I've also narrowed the orginal dataset to only include students):
- Which factors correlate strongest with depression among students.
- How mental heatlth concerns vary by factors like gender, age, 


In [ ]:
import pandas as pd

original_df = pd.read_csv("student_depression_dataset.csv")

#df for just students 
df = original_df[original_df.Profession=="Student"]
df

In [ ]:
print("Double checking: number of non-students =",len(df[df.Profession!="Student"]))
print("Number of rows, columns:", df.shape)
print("The different columns:", df.columns.unique())
print("Missing Values:\n",df.isnull().sum())

## Section 2: Exploratory Data Analysis
For this dataset, descriptive statistics like measure of center (mean,median) and measures of spread (std, min, max, percentiles) are especially useful here because they will help to understand general trends in mental health factors. Knowing mean or median levels of quantitative data like age and academic pressure will show us what's considered "normal" within this group while the spread will show us how the students' experiences differ. Measuring the totals of the subgroups in qualitative data columns like gender and degree will help understand the makeup of the population to find if particular groups are more likely to say they have depression symptoms.

In [ ]:
print("Average/median Age:", df.Age.mean()," /", df.Age.median())
print("Spread/min/max Age:", df.Age.std(),"/",df.Age.min(),"/",df.Age.max(),"\n")

print("Average/median Academic Pressure:",df["Academic Pressure"].mean()," /",df["Academic Pressure"].median())
print("Spread/min/max Academic Pressure:",df["Academic Pressure"].std(),"/",df["Academic Pressure"].min(),"/",df["Academic Pressure"].max(),"\n")

print("Average/median Work Pressure:",df["Work Pressure"].mean(),"/",df["Work Pressure"].median())
print("Spread/min/max Work Pressure:",df["Work Pressure"].std()," /",df["Work Pressure"].min(),"/",df["Work Pressure"].max(),"\n")

print("Average/median CumGPA:",df.CGPA.mean(),"   /",df.CGPA.median())
print("Spread/min/max CumGPA:", df.CGPA.std(),"/",df.CGPA.min(),"/",df.CGPA.max(),"\n")

print("Average/median Study Satisfaction:",df["Study Satisfaction"].mean(),"  /",df["Study Satisfaction"].median())
print("Spread/min/max Study Satisfaction:",df["Study Satisfaction"].std(),"/",df["Study Satisfaction"].min(),"/",df["Study Satisfaction"].max(),"\n")

print("Average/median Job Satisfaction:",df["Job Satisfaction"].mean()," /",df["Job Satisfaction"].median())
print("Spread/min/max Job Satisfaction:",df["Job Satisfaction"].std(),"  /",df["Job Satisfaction"].min(),"/",df["Job Satisfaction"].max(),"\n")

print("Average/median Work/Study Hours:",df["Work/Study Hours"].mean(),"  /",df["Work/Study Hours"].median())
print("Spread/min/max Work/Study Hours:",df["Work/Study Hours"].std(),"/",df["Work/Study Hours"].min(),"/",df["Work/Study Hours"].max(),"\n")


df["Financial Stress"]= df["Financial Stress"].replace("?",0).astype(float).astype(int)
print("Average/median Financial Stress:",df["Financial Stress"].mean(),"  /",df["Financial Stress"].median())
print("Spread/min/max Financial Stress:",df["Financial Stress"].std(),"/",df["Financial Stress"].min(),"/",df["Financial Stress"].max(),"\n")

### Explanation of Exploratory Analysis:
The means all seem to line up fairly accurately with their medians showing there shouldn't be too many outliers or they don't affect the data that much because the sample is so large. "Study hours" and "age" are the only variables where the mean and median are NOT within .2 of each other, and as we can see they also have the highest spreads meaning their might be more variation in those responses that need further analysis. We also see that mean and median "work pressure" and "job satisfaction" are relatively 0, this is not going to help my analysis at face value. This likely occured becuase most of the students in this sample probably aren't working and I'll need to stratify the data to further analyze this variable. 

## Section 3: Exploratory Data Visualization


In [ ]:
df.Age.plot.box(title="Distribution of Age")

In [ ]:
df[["Work/Study Hours","CGPA"]].plot.box(title="Distribution of Work/Study hours")

In [ ]:
df[["Academic Pressure","Work Pressure","Study Satisfaction","Job Satisfaction","Financial Stress"]].plot.box(figsize=(9,4),ylabel="responses on 0-5 scale",title="Distributions of Academic/Work Pressure, Study Satisfaction, Job Satisfaction, and Financial Stress")

### Explanation of Exploratory Visualization:
In the first graph, the ages of the sample have a decent amount of outliers so it might be worth removing those to get a more accurate representation of the data. In the second graph we can see the distribution of Work/Study hours very cleary while the outlier in the Cumulative GPA graph makes it very hard to estimate percentiles. As we can see in the third graph nearly 100% of the samples responded with 0 on questions relating to work. We can also see that the spread for Academic Pressure, Study Satistfaction, and Financial stress are all roughly the same.

## Section 4: Data Science
### Observation:

Since I only have data (other than 0) for a few samples of variables involving having a job, it's safe to say we can't determine any useful information about depression based on work pressure or job satisfaction. With the spread of Academic Pressure, Study Satisfaction, and Financial Stress being roughly the same, this could indicate correlation between these variables.

### Ideas for further analysis:

It might be worth narrowing the data to the middle 50% of the ages to get data from those closer to the age of the average college student. This would involve calculating the IQR of ages and removing the samples with ages outside its range. I could also find the probabilty someone reports having depression given they are sleeping less than 5 hours. I will likely repeat this process with other variables that I might think have significant effect on depression to find which one has a greater affect on depression. I can then compare these probabilities with their "less risky" counterpart (probability of depression given 8 hours of sleep, etc.) to see if people are reporting depression despite the high or low risk of the variable. This would involve calculating the different conditional probabilities using P(A|B)=P(A and B)/P(B) and comparing them to one another.

## Section 5: Data Science
#### The question I’ve chosen to answer: 
Is the probability of reporting depression higher among students who sleep less than 7 hours compared to those who sleep 7 or more hours?

To answer this, I'm going to filter the dataset into two groups based on sleep duration (<7 hours and >7 hours). Then, I’ll find the proportion of students in each group who report depression. Using these values, I'll perform a two sample, one-tailed, hypothesis test to compare the depression rates between the two sleep groups. Specifically, my null hypothesis is that there's no difference in depression rate, while my alternative hypothesis is that students who sleep less than 7 hours have a higher probability of reporting depression than those who sleep 7 or more hours.

In [ ]:
df["Sleep Duration"].unique()

In [ ]:
df_LessThan7 = df[(df["Sleep Duration"]=="'Less than 5 hours'")|(df["Sleep Duration"]=="'5-6 hours'")]
df_LessThan7_WithDepression = df[((df["Sleep Duration"]=="'Less than 5 hours'")|(df["Sleep Duration"]=="'5-6 hours'"))&(df.Depression==0)]
p_depression_given_LessThan7 = len(df_LessThan7_WithDepression)/len(df_LessThan7)
print("P(Depression | Sleep < 7hours):",p_depression_given_LessThan7)

In [ ]:
df_MoreThan6 = df[(df["Sleep Duration"]=="'More than 8 hours'")|(df["Sleep Duration"]=="'7-8 hours'")]
df_MoreThan6_WithDepression = df[((df["Sleep Duration"]=="'More than 8 hours'")|(df["Sleep Duration"]=="'7-8 hours'"))&(df.Depression==0)]
p_depression_given_MoreThan6 = len(df_MoreThan6_WithDepression)/len(df_MoreThan6)
print("P(Depression | Sleep >= 7hours):",p_depression_given_MoreThan6)

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

n1 = len(df_LessThan7)
n2 = len(df_MoreThan6)
x1 = len(df_LessThan7_WithDepression)
x2 = len(df_MoreThan6_WithDepression)

z, pValue = proportions_ztest([x1,x2],[n1,n2],alternative="larger")
print("Z Statistic: ",z)
print("p-value, one-sided: ",pValue)

z, pValue = proportions_ztest([x1,x2],[n1,n2],alternative="two-sided")
print("p-value, two-sided: ",pValue)

print("\nConclusion: There is not significant evidence to suggest that students who sleep less than seven\nhours will have a higher reported depression rates than students sleeping 7 or more hours.")

## Section 6: A Different Data Visualization


In [ ]:
data = {"Sleep":["<7 hours",">=7 hours"],"Proportion with Depression":[p_depression_given_LessThan7,p_depression_given_MoreThan6]}
pdf = pd.DataFrame(data)

pdf.plot.bar(x="Sleep",y="Proportion with Depression",ylabel="Percentage",title="Proportion of Students Reporting Depression by Sleep Duration",figsize=(6,6),yticks=[0,.05,.1,.15,.2,.25,.3,.35,.4,.45])

## Section 7: Overall Summary


For this project, I explored the relationship between sleep duration and depression among students using a public dataset. I grouped students into two categories: those who sleep less than 7 hours per night and those who sleep 7 or more. I then found the proportion of students in each group who reported depression given their general sleep schedule. Surprisingly, I found that 38.76% of students who sleep less than 7 hours reported depression, while 44.42% of students who sleep 7 or more hours reported depression. 


To visualize this comparison, I created a simple bar chart showing the proportion of students with depression in each sleep group. The visualization clearly showed a higher reported depression rate among the longer-sleeping students. To formally test this difference, I performed a two-sample, one-sided, z-test with the hypothesis that students sleeping less than 7 hours would have a higher depression rate. The test resulted in a z statistic of -9.57 and a p-value of 1, meaning the data did not support my hypothesis. I can conclude that students who sleep more may actually report depression more frequently. One possible explanation could be that students experiencing depression sleep more than usual, making sleep duration a symptom rather than a cause.